In [1]:
# Cell 1: Imports
from playwright.async_api import async_playwright
import asyncio
import re
from dataclasses import dataclass
from pydantic import BaseModel, Field
from typing import Optional, List
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
import requests
from bs4 import BeautifulSoup
import time

load_dotenv(override=True)

True

In [2]:
# Cell 2: ScrapedBestBuyDeal - Dữ liệu thô từ Playwright
class ScrapedBestBuyDeal:
    """
    A class to represent a Deal scraped from BestBuy using Playwright.
    Similar to ScrapedDeal but for BestBuy products.
    """
    
    title: str
    brand: Optional[str]
    price: float
    features: str
    url: str
    
    def __init__(self, title: str, brand: Optional[str], price: float, features: str, url: str):
        """
        Initialize with scraped data from BestBuy product page.
        """
        self.title = title[:200] if title else "Unknown"
        self.brand = brand.strip() if brand else None
        self.price = price
        self.features = features[:1500] if features else ""
        self.url = url
    
    def __repr__(self) -> str:
        """Return a short string description."""
        return f"<{self.title[:50]}... | ${self.price}>"
    
    def describe(self) -> str:
        """
        Return a longer string to describe this deal for use in calling a model.
        Similar to ScrapedDeal.describe() format.
        """
        parts = [f"Title: {self.title}"]
        
        if self.brand:
            parts.append(f"Brand: {self.brand}")
        
        parts.append(f"Price: ${self.price:.2f}")
        
        if self.features and len(self.features) > 10:
            parts.append(f"Features: {self.features.strip()}")
        
        parts.append(f"URL: {self.url}")
        
        return "\n".join(parts)

In [3]:
# Cell 3: Scrape BestBuy products → List[ScrapedBestBuyDeal]
async def scrape_bestbuy_products(urls: List[str], headless: bool = False) -> List[ScrapedBestBuyDeal]:
    """
    Scrape BestBuy products and return as List[ScrapedBestBuyDeal].
    
    Args:
        urls: List of BestBuy product URLs (preferably sale items)
        headless: Run browser in headless mode
        
    Returns:
        List[ScrapedBestBuyDeal] - Raw scraped data
    """
    scraped_deals = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox']
        )
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080}
        )
        page = await context.new_page()
        
        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] Scraping...")
            
            try:
                await page.goto(url, timeout=60000, wait_until="domcontentloaded")
                await page.wait_for_timeout(2000)
                
                # Extract Title
                title_elem = page.locator("h1.h4")
                title = await title_elem.text_content() if await title_elem.count() > 0 else "Unknown"
                title = title.strip() if title else "Unknown"
                
                # Extract Brand
                brand_elem = page.locator('div[data-component-name="ProductHeader"] a.c-button-link')
                brand = await brand_elem.first.text_content() if await brand_elem.count() > 0 else None
                brand = brand.strip() if brand else None
                
                # Extract Sale Price
                price_elem = page.locator('[data-testid="price-block-customer-price"] span')
                price_text = await price_elem.first.text_content() if await price_elem.count() > 0 else "$0"
                price_match = re.search(r'[\d,]+\.?\d*', price_text.replace(',', ''))
                price = float(price_match.group()) if price_match else 0.0
                
                # Click Features button and extract
                features = ""
                features_btn = page.locator('button:has(h3:text("Features"))')
                
                if await features_btn.count() > 0:
                    await features_btn.first.click()
                    try:
                        await page.locator('[data-testid="brix-sheet-content"]').wait_for(timeout=5000)
                        features_elem = page.locator('[data-testid="brix-sheet-content"]')
                        features = await features_elem.first.text_content() or ""
                    except:
                        pass
                    await page.keyboard.press("Escape")
                    await page.wait_for_timeout(500)
                
                # Create ScrapedBestBuyDeal
                deal = ScrapedBestBuyDeal(
                    title=title,
                    brand=brand,
                    price=price,
                    features=features,
                    url=url
                )
                scraped_deals.append(deal)
                print(f"  ✓ {deal}")
                
            except Exception as e:
                print(f"  ✗ Error: {e}")
                continue
        
        await browser.close()
    
    return scraped_deals

In [4]:
sale_urls = ['https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K',
 'https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW',
 'https://www.bestbuy.com/product/tcl-65-class-qm8k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQZ3T',
 'https://www.bestbuy.com/product/samsung-65-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5VV',
 'https://www.bestbuy.com/product/tcl-65-class-qm5k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQWZ4',
 'https://www.bestbuy.com/product/tcl-50-class-qm5k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTT2Y8',
 'https://www.bestbuy.com/product/tcl-55-qm6k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQW5C',
 'https://www.bestbuy.com/product/samsung-70-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2VF78',
 'https://www.bestbuy.com/product/tcl-55-class-qm5k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQWZR',
 'https://www.bestbuy.com/product/tcl-40-class-q3k-series-1080p-fhd-qled-smart-tv-with-google-tv-2025/J36QYTQZ79',
 'https://www.bestbuy.com/product/tcl-65-class-qm6k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQW5K',
 'https://www.bestbuy.com/product/tcl-85-class-qm5k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQW4Z',
 'https://www.bestbuy.com/product/tcl-85-class-qm6k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQWFL',
 'https://www.bestbuy.com/product/tcl-98-class-qm8k-series-4k-uhd-hdr-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQZXS']

In [5]:
test_urls = sale_urls[:3]  # Lấy 3 URLs đầu tiên để test


In [6]:
test_urls

['https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K',
 'https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW',
 'https://www.bestbuy.com/product/tcl-65-class-qm8k-series-4k-uhd-qd-mini-led-smart-tv-with-google-tv-2025/J36QYTQZ3T']

In [7]:
# Cell 4: Test scrape 3 URLs
# Giả sử bạn đã có sale_urls từ bước trước
test_urls = sale_urls[:3]  # Lấy 3 URLs đầu tiên để test

print(f"Testing with {len(test_urls)} URLs...\n")

scraped_deals = await scrape_bestbuy_products(test_urls, headless=False)

print(f"\n{'='*60}")
print(f"Scraped {len(scraped_deals)} deals")
print(f"{'='*60}")

for i, deal in enumerate(scraped_deals, 1):
    print(f"\n--- ScrapedBestBuyDeal {i} ---")
    print(deal.describe())

Testing with 3 URLs...

[1/3] Scraping...
  ✓ <Westinghouse - 24” Class Smart TV, HD Xumo TV with... | $79.99>
[2/3] Scraping...
  ✓ <Samsung - 55" Class U7900 Series UHD 4K Smart Tize... | $279.99>
[3/3] Scraping...
  ✓ <TCL - 65" Class QM8K Series 4K UHD QD-Mini LED Sma... | $999.99>

Scraped 3 deals

--- ScrapedBestBuyDeal 1 ---
Title: Westinghouse - 24” Class Smart TV, HD Xumo TV with Voice Remote, Flat Screen LED Television
Brand: Westinghouse
Price: $79.99
Features: Westinghouse - 24” Class Smart TV, HD Xumo TV with Voice Remote, Flat Screen LED TelevisionRating 4.7 out of 5 stars with 130 reviews4.7(130 reviews)The Westinghouse 24-inch Smart TV delivers crisp HD entertainment in a sleek flat-screen LED design, perfect for any room in your home. Powered by Xumo TV, it offers easy access to live channels, movies, and streaming apps, all controlled with the included voice remote for effortless navigation. With built-in Wi-Fi and mobile connectivity, you can stream your favorite con

In [10]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
    Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
    Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
    
    **CRITICAL PRICING RULES:**
    1. **"Off" / "Reduced by":** Be careful with products described as "$XXX off" or "reduced by $XXX" - this isn't the actual price. Only respond when the final checkout price is explicitly stated.
    2. **"Up to":** EXCLUDE general sales events labeled as "Up to 70% off" or "Up to $1,800 off" unless a specific individual item with a specific numeric price is clearly listed.
    3. **Trade-ins:** EXCLUDE deals that require a trade-in (e.g., "$700 off w/ trade-in"). We only want the direct purchase price without exchanging an old device.
    4. **Upgrades:** If a deal says "512GB for 256GB price", extract it, but ensure the 'price' field is the actual dollar amount shown, not the value of the upgrade.
    """

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
    You should rephrase the description to be a summary of the product itself, not the terms of the deal.
    Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
    
    **STRICT FILTERING CRITERIA:**
    - **Ignore Trade-ins:** Do not include prices that depend on "w/ trade-in" or "with eligible plan".
    - **Ignore "Up to" ranges:** Do not include generic sales like "Up to 70% off" or "Up to $1,800 off" if no specific item price is visible.
    - **Clarify "Free Upgrade":** For deals like "512GB for 256GB price", record the price required to purchase the item.

    Deals:

    """

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."


In [12]:
def make_user_prompt(self, scraped) -> str:
        """
        Create a user prompt for OpenAI based on the scraped deals provided
        """
        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += "\n\n".join([scrape.describe() for scrape in scraped])
        user_prompt += self.USER_PROMPT_SUFFIX
        return user_prompt

In [11]:
# Cell 5: Xem prompt sẽ gửi cho GPT-5-mini
print("=" * 60)
print("PROMPT SẼ GỬI CHO GPT-5-mini (tương tự ScannerAgent)")
print("=" * 60)

user_prompt = "Respond with the 5 most promising deals from this list...\n\n"
user_prompt += "\n\n".join([deal.describe() for deal in scraped_deals])

print(user_prompt[:2000])
print("\n... (truncated)")

PROMPT SẼ GỬI CHO GPT-5-mini (tương tự ScannerAgent)
Respond with the 5 most promising deals from this list...

Title: Westinghouse - 24” Class Smart TV, HD Xumo TV with Voice Remote, Flat Screen LED Television
Brand: Westinghouse
Price: $79.99
Features: Westinghouse - 24” Class Smart TV, HD Xumo TV with Voice Remote, Flat Screen LED TelevisionRating 4.7 out of 5 stars with 130 reviews4.7(130 reviews)The Westinghouse 24-inch Smart TV delivers crisp HD entertainment in a sleek flat-screen LED design, perfect for any room in your home. Powered by Xumo TV, it offers easy access to live channels, movies, and streaming apps, all controlled with the included voice remote for effortless navigation. With built-in Wi-Fi and mobile connectivity, you can stream your favorite content straight from your devices, while Apple HomeKit compatibility lets you integrate the TV seamlessly into your smart home setup. Compact yet powerful, this smart television combines convenience, connectivity, and clear 

In [13]:
# Cell 6: Import OpenAI
from openai import OpenAI

In [31]:
# Cell 7: Import Deal, DealSelection, Opportunity từ hệ thống
# KHÔNG định nghĩa lại - sử dụng class có sẵn

from price_agents.deals import Deal, DealSelection, Opportunity

print("Imported Deal, DealSelection, Opportunity from price_agents.deals")

Imported Deal, DealSelection, Opportunity from price_agents.deals


In [32]:
# Cell 8: BestBuyScannerAgent - Chọn 5 deals tốt nhất bằng GPT-5-mini
class BestBuyScannerAgent:
    """
    Agent that uses GPT-5-mini to select the best 5 deals from scraped BestBuy products.
    Similar to ScannerAgent but for BestBuy data.
    """
    
    MODEL = "gpt-5-mini"
    
    SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
    Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description.
    Most important is that you respond with the 5 deals that have the most detailed product description with price.
    
    **IMPORTANT:**
    1. Focus on the product features and specifications, not sales terms.
    2. The product_description should be a 3-4 sentence summary of the product itself.
    3. Price must be greater than 0.
    4. Keep the original URL exactly as provided.
    """
    
    USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
    You should rephrase the description to be a summary of the product itself, not the terms of the deal.
    Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
    
    Deals:
    
    """
    
    USER_PROMPT_SUFFIX = "\n\nInclude up to 5 deals, no more."
    
    def __init__(self):
        """Initialize with OpenAI client."""
        self.openai = OpenAI()
        print("BestBuyScannerAgent initialized")
    
    def make_user_prompt(self, scraped_deals: List[ScrapedBestBuyDeal]) -> str:
        """Create user prompt from scraped deals."""
        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += "\n\n".join([deal.describe() for deal in scraped_deals])
        user_prompt += self.USER_PROMPT_SUFFIX
        return user_prompt
    
    def scan(self, scraped_deals: List[ScrapedBestBuyDeal]) -> Optional[DealSelection]:
        """
        Call GPT-5-mini to select the best 5 deals with good descriptions and prices.
        
        Args:
            scraped_deals: List of ScrapedBestBuyDeal from Playwright scraping
            
        Returns:
            DealSelection with up to 5 best deals, or None if no valid deals
        """
        if not scraped_deals:
            print("No deals to scan")
            return None
        
        # Filter deals with price > 0
        valid_deals = [d for d in scraped_deals if d.price > 0]
        if not valid_deals:
            print("No deals with valid price")
            return None
        
        user_prompt = self.make_user_prompt(valid_deals)
        
        print(f"Calling GPT-5-mini with {len(valid_deals)} deals...")
        
        result = self.openai.chat.completions.parse(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            response_format=DealSelection,
        )
        
        selection = result.choices[0].message.parsed
        
        # Filter out any deals with price <= 0
        selection.deals = [deal for deal in selection.deals if deal.price > 0]
        
        print(f"GPT-5-mini selected {len(selection.deals)} deals")
        
        return selection

In [33]:
# Cell 9: Test BestBuyScannerAgent với các deals đã scrape
scanner = BestBuyScannerAgent()

# Sử dụng scraped_deals từ Cell 4 (3 deals test)
deal_selection = scanner.scan(scraped_deals)

print(f"\n{'='*60}")
print(f"✅ DealSelection with {len(deal_selection.deals)} deals:")
print(f"{'='*60}")

for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n--- Deal {i} ---")
    print(f"📝 Description: {deal.product_description[:200]}...")
    print(f"💰 Price: ${deal.price}")
    print(f"🔗 URL: {deal.url}")

BestBuyScannerAgent initialized
Calling GPT-5-mini with 3 deals...
GPT-5-mini selected 3 deals

✅ DealSelection with 3 deals:

--- Deal 1 ---
📝 Description: This 24-inch Westinghouse LED smart TV delivers 720p HD resolution with progressive scan for smoother motion and reduced flicker. It runs the Xumo TV platform for access to live channels and streaming...
💰 Price: $79.99
🔗 URL: https://www.bestbuy.com/product/westinghouse-24-class-smart-tv-hd-xumo-tv-with-voice-remote-flat-screen-led-television/J3LL89C69K

--- Deal 2 ---
📝 Description: The Samsung 55" U7900 Series is a 4K UHD smart TV powered by the Crystal Processor 4K for vivid color reproduction and upscaling of lower‑resolution content. It features a sleek MetalStream single‑she...
💰 Price: $279.99
🔗 URL: https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW

--- Deal 3 ---
📝 Description: The TCL 65" QM8K Series is a premium QD‑Mini LED 4K TV that uses the Halo Control System and adv

In [18]:
# Cell 10: Full pipeline test với nhiều URLs hơn
print("=" * 60)
print("FULL PIPELINE TEST")
print("=" * 60)

# Scrape tất cả sale_urls (hoặc giới hạn 10 đầu tiên để test)
test_urls = sale_urls[:10]  # Giới hạn 10 URLs

print(f"\nStep 1: Scraping {len(test_urls)} URLs with Playwright...")
all_scraped = await scrape_bestbuy_products(test_urls, headless=False)

print(f"\nStep 2: Selecting top 5 deals with GPT-5-mini...")
final_selection = scanner.scan(all_scraped)

print(f"\n{'='*60}")
print(f"✅ FINAL RESULT: {len(final_selection.deals)} selected deals")
print(f"{'='*60}")

for i, deal in enumerate(final_selection.deals, 1):
    print(f"\n🏷️ Deal {i}:")
    print(f"   {deal.product_description[:150]}...")
    print(f"   💰 ${deal.price}")

FULL PIPELINE TEST

Step 1: Scraping 10 URLs with Playwright...
[1/10] Scraping...
  ✓ <Westinghouse - 24” Class Smart TV, HD Xumo TV with... | $79.99>
[2/10] Scraping...
  ✓ <Samsung - 55" Class U7900 Series UHD 4K Smart Tize... | $279.99>
[3/10] Scraping...
  ✓ <TCL - 65" Class QM8K Series 4K UHD QD-Mini LED Sma... | $999.99>
[4/10] Scraping...
  ✓ <Samsung - 65" Class U7900 Series UHD 4K Smart Tize... | $329.99>
[5/10] Scraping...
  ✓ <TCL - 65" Class QM5K Series 4K UHD QD-Mini LED Sma... | $0.0>
[6/10] Scraping...
  ✓ <TCL - 50" Class QM5K Series 4K UHD HDR QD-Mini LED... | $299.99>
[7/10] Scraping...
  ✓ <TCL - 55" QM6K Series 4K UHD HDR QD Mini LED Smart... | $449.99>
[8/10] Scraping...
  ✓ <Samsung - 70" Class U7900 Series UHD 4K Smart Tize... | $399.99>
[9/10] Scraping...
  ✓ <TCL - 55" Class QM5K Series 4K UHD HDR QD-Mini LED... | $329.99>
[10/10] Scraping...
  ✓ <TCL - 40" Class Q3K Series 1080P FHD QLED Smart TV... | $149.99>

Step 2: Selecting top 5 deals with GPT-5-mini...

In [34]:
# Cell 11: Import EnsembleAgent từ hệ thống hiện tại
import os
import sys

# ⚠️ QUAN TRỌNG: Chuyển working directory về segment4
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
print(f"Working directory: {os.getcwd()}")

# Thêm path để import modules
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

import chromadb
from price_agents.ensemble_agent import EnsembleAgent
from price_agents.deals import Opportunity

# Khởi tạo ChromaDB và EnsembleAgent
DB_PATH = "products_vectorstore"  # Relative path (vì đã chdir)
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')

print("Initializing EnsembleAgent...")
ensemble = EnsembleAgent(collection)
print("EnsembleAgent ready!")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
Initializing EnsembleAgent...
EnsembleAgent ready!


In [35]:
# Cell 12: Estimate giá trị thực và tạo Opportunity
def estimate_deals(deal_selection: DealSelection, ensemble: EnsembleAgent) -> List[Opportunity]:
    """
    Use EnsembleAgent to estimate true value for each deal.
    
    Args:
        deal_selection: DealSelection from BestBuyScannerAgent
        ensemble: EnsembleAgent instance
        
    Returns:
        List[Opportunity] sorted by discount (highest first)
    """
    opportunities = []
    
    for i, deal in enumerate(deal_selection.deals, 1):
        print(f"\n[{i}/{len(deal_selection.deals)}] Estimating: {deal.product_description[:50]}...")
        
        # Gọi EnsembleAgent để dự đoán giá trị thực
        estimate = ensemble.price(deal.product_description)
        
        # Tính discount
        discount = estimate - deal.price
        
        # Tạo Opportunity
        opportunity = Opportunity(
            deal=deal,
            estimate=estimate,
            discount=discount
        )
        opportunities.append(opportunity)
        
        print(f"   💰 Sale Price: ${deal.price:.2f}")
        print(f"   📊 Estimate:   ${estimate:.2f}")
        print(f"   🏷️  Discount:   ${discount:.2f}")
    
    # Sắp xếp theo discount giảm dần
    opportunities.sort(key=lambda x: x.discount, reverse=True)
    
    return opportunities

In [36]:
# Cell 13: Test estimate với 3 deals đã có
print("=" * 60)
print("ESTIMATING TRUE VALUE FOR EACH DEAL")
print("=" * 60)

opportunities = estimate_deals(deal_selection, ensemble)

print(f"\n{'='*60}")
print(f"✅ OPPORTUNITIES RANKED BY DISCOUNT:")
print(f"{'='*60}")

for i, opp in enumerate(opportunities, 1):
    print(f"\n🏆 Rank {i}:")
    print(f"   📝 {opp.deal.product_description[:100]}...")
    print(f"   💵 Sale Price:     ${opp.deal.price:.2f}")
    print(f"   📊 Estimated Value: ${opp.estimate:.2f}")
    print(f"   🔥 DISCOUNT:        ${opp.discount:.2f}")
    print(f"   🔗 {opp.deal.url}")

ESTIMATING TRUE VALUE FOR EACH DEAL

[1/3] Estimating: This 24-inch Westinghouse LED smart TV delivers 72...
   💰 Sale Price: $79.99
   📊 Estimate:   $155.95
   🏷️  Discount:   $75.96

[2/3] Estimating: The Samsung 55" U7900 Series is a 4K UHD smart TV ...
   💰 Sale Price: $279.99
   📊 Estimate:   $593.48
   🏷️  Discount:   $313.49

[3/3] Estimating: The TCL 65" QM8K Series is a premium QD‑Mini LED 4...
   💰 Sale Price: $999.99
   📊 Estimate:   $1160.44
   🏷️  Discount:   $160.45

✅ OPPORTUNITIES RANKED BY DISCOUNT:

🏆 Rank 1:
   📝 The Samsung 55" U7900 Series is a 4K UHD smart TV powered by the Crystal Processor 4K for vivid colo...
   💵 Sale Price:     $279.99
   📊 Estimated Value: $593.48
   🔥 DISCOUNT:        $313.49
   🔗 https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW

🏆 Rank 2:
   📝 The TCL 65" QM8K Series is a premium QD‑Mini LED 4K TV that uses the Halo Control System and advance...
   💵 Sale Price:     $999.99
   📊 Estimated 

In [37]:
# Cell 14: Tìm Best Deal
DEAL_THRESHOLD = 50  # Threshold từ PlanningAgent

print("=" * 60)
print(f"DEALS WITH DISCOUNT > ${DEAL_THRESHOLD}")
print("=" * 60)

good_deals = [opp for opp in opportunities if opp.discount > DEAL_THRESHOLD]

if good_deals:
    best = good_deals[0]
    print(f"\n🎉 BEST DEAL FOUND!")
    print(f"   📝 {best.deal.product_description}")
    print(f"   💵 Sale Price:     ${best.deal.price:.2f}")
    print(f"   📊 Estimated Value: ${best.estimate:.2f}")
    print(f"   🔥 YOU SAVE:        ${best.discount:.2f}")
    print(f"   🔗 {best.deal.url}")
else:
    print(f"\n❌ No deals with discount > ${DEAL_THRESHOLD}")
    print(f"   Best available discount: ${opportunities[0].discount:.2f}" if opportunities else "No deals")

DEALS WITH DISCOUNT > $50

🎉 BEST DEAL FOUND!
   📝 The Samsung 55" U7900 Series is a 4K UHD smart TV powered by the Crystal Processor 4K for vivid color reproduction and upscaling of lower‑resolution content. It features a sleek MetalStream single‑sheet metal design with a slim bezel that minimizes distractions and enhances room aesthetics. The Tizen smart platform provides thousands of on‑demand apps and content, while Samsung Knox security protects the TV and connected IoT devices from unauthorized apps and threats.
   💵 Sale Price:     $279.99
   📊 Estimated Value: $593.48
   🔥 YOU SAVE:        $313.49
   🔗 https://www.bestbuy.com/product/samsung-55-class-u7900-series-uhd-4k-smart-tizen-tv-2025/J3ZYG2V5FW
